Copyright (c) 2026 Gabor Seljan. All rights reserved.

Licensed under the MIT License.

# SeedClass

SeedClass is an experimental machine learning project employing binary classification to predict whether fuzzing a specific seed file will lead to a vulnerability.

In [ ]:
import os
import onnx
import config
import tf2onnx
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
import plotly.express as px
import matplotlib.pyplot as plt

from tensorflow.keras import Input
from tensorflow.keras.metrics import AUC, BinaryAccuracy, Recall, Precision
from tensorflow.keras.metrics import TruePositives, TrueNegatives
from tensorflow.keras.metrics import FalsePositives, FalseNegatives
from tensorflow.keras.regularizers import L1, L2
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard

from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import train_test_split

from shutil import copy
from utils import load_data
from utils import take_rows
from utils import plot_prc
from mpl_toolkits.mplot3d import Axes3D

if config.VALIDATION:
    from utils import plot_cross_val_essential_metrics
    from utils import plot_cross_val_confusion_metrics
else:
    from utils import plot_essential_metrics
    from utils import plot_confusion_metrics

if config.SEED is not None:
    tf.keras.utils.set_random_seed(config.SEED)

In [ ]:
dd = {}
dd = load_data(dd, os.path.join('data', 'training', config.DATASET, 'valid.zip'), config.LABEL_NEGATIVE)
dd = load_data(dd, os.path.join('data', 'training', config.DATASET, 'crash.zip'), config.LABEL_POSITIVE)

In [ ]:
df = pd.DataFrame.from_dict(dd, orient='index').add_prefix('f')
df.info()
df.describe().transpose()

In [ ]:
plt.figure(figsize=(8, 6))
sns.countplot(x='f0', data=df)
plt.xlabel('Label')
plt.xticks(ticks=[0, 1], labels=['Valid', 'Crash'])
plt.ylabel('Samples')
plt.title('Distribution of Labels')
plt.savefig(os.path.join('figures', 'learning', config.DATASET, 'countplot.png'), bbox_inches='tight')

In [ ]:
corr = df.corr()
plt.figure(figsize=[8, 6])
ax = corr['f0'][1:-1].sort_values().plot(kind='bar')
ax.tick_params(axis='x', bottom=False, labelbottom=False)
plt.xlabel('Features')
plt.ylabel('Correlation')
plt.title('Feature Correlation with Target Variable')
plt.savefig(os.path.join('figures', 'learning', config.DATASET, 'correlation.png'), bbox_inches='tight')

In [ ]:
plt.figure(figsize=[8, 6])
sns.heatmap(corr)
plt.xlabel('Features')
plt.ylabel('Features')
plt.title('Feature Correlation Heatmap')
plt.savefig(os.path.join('figures', 'learning', config.DATASET, 'heatmap.png'), bbox_inches='tight')

In [ ]:
mask = (abs(corr) > 0.5) & (abs(corr) != 1)
corr.where(mask).stack().sort_values()

In [ ]:
X = df.drop('f0', axis=1).values
y = df['f0'].values

In [ ]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(X, y, test_size=config.TEST_SIZE, shuffle=True, random_state=config.SEED, stratify=y)

In [ ]:
plot_scaler = MinMaxScaler(feature_range=(0, config.FEATURE_RANGE_MAX))
X_train = pd.DataFrame(plot_scaler.fit_transform(X_train_raw)).values

In [ ]:
pca = PCA()
pca.fit_transform(X_train)
cumsum = np.cumsum(pca.explained_variance_ratio_)
d = np.argmax(cumsum >= 0.95) + 1
print(f'Min {d} dimensions required for 95% variance')

In [ ]:
N=3
pca = PCA(n_components=N)
X_pca = pca.fit_transform(X_train)
X_pca_df = pd.DataFrame(X_pca, columns = ['PC1', 'PC2', 'PC3'])
fig = px.scatter_3d(X_pca_df, x='PC1', y='PC2', z='PC3', size_max=18, opacity=0.7).update_traces(marker = dict(color = y_train))
fig.update_layout(margin=dict(l=0, r=0, b=0, t=0))
fig.show()
print(f'Variance: {sum(pca.explained_variance_ratio_) * 100:.2f}%')

In [ ]:
N=3
pca = PCA(n_components=N)
X_pca = pca.fit_transform(X_train)
variance = sum(pca.explained_variance_ratio_) * 100
fig = plt.figure(figsize=[8, 6])
ax = fig.add_subplot(111, projection='3d')
ax.set_box_aspect([0.849, 1, 1])
for i in range(0, N-2, 3):
    ax.scatter(X_pca[:,i+1], X_pca[:,i], X_pca[:,i+2], c=y_train)
ax.text2D(0.05, 0.95, f'Variance: {variance:.2f}%', transform=ax.transAxes, fontsize=10)
ax.set_xlabel('PC2')
ax.set_ylabel('PC1')
ax.set_zlabel('PC3')
plt.title(f'{N} Component PCA')
fig.savefig(os.path.join('figures', 'learning', config.DATASET, 'pca-3.png'), bbox_inches='tight')

In [ ]:
N=2
pca = PCA(n_components=N)
X_pca = pca.fit_transform(X_train)
variance = sum(pca.explained_variance_ratio_) * 100
plt.figure(figsize=[8, 6])
for i in range(0, N-1, 2):
    plt.scatter(X_pca[:, i], X_pca[:, i+1], c=y_train)
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title(f'{N} Component PCA')
plt.text(0.05, 0.95, f'Variance: {variance:.2f}%', transform=plt.gca().transAxes, fontsize=10, verticalalignment='top')
plt.savefig(os.path.join('figures', 'learning', config.DATASET, 'pca-2.png'), bbox_inches='tight')

In [ ]:
N=2
tsne = TSNE(n_components=N, perplexity=30, verbose=1)
X_tsne = tsne.fit_transform(X_train)
plt.figure(figsize=[8, 6])
plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=y_train)
plt.xlabel('t-SNE C1')
plt.ylabel('t-SNE C2')
plt.title(f'{N} Component TSNE')
plt.savefig(os.path.join('figures', 'learning', config.DATASET, 'tsne.png'), bbox_inches='tight')

In [ ]:
def build_model():
    model = Sequential()
    model.add(Input(shape=(config.NUM_FEATURES,)))
    model.add(Dense(config.NUM_FEATURES, activation='relu', kernel_regularizer=L2(0.00001) if config.REGULARIZATION else None))
    model.add(Dropout(0.5))
    model.add(Dense(config.NUM_FEATURES, activation='relu', kernel_regularizer=L2(0.00001) if config.REGULARIZATION else None))
    model.add(Dropout(0.5))
    model.add(Dense(1, activation='sigmoid'))

    model.compile(
        optimizer=Adam(learning_rate=config.LEARNING_RATE),
        loss='binary_crossentropy',
        metrics=[BinaryAccuracy(name='accuracy', threshold=config.THRESHOLD),
            Recall(thresholds=config.THRESHOLD, name='recall'),
            Precision(thresholds=config.THRESHOLD, name='precision'),
            TruePositives(thresholds=config.THRESHOLD, name='tp'),
            TrueNegatives(thresholds=config.THRESHOLD, name='tn'),
            FalsePositives(thresholds=config.THRESHOLD, name='fp'),
            FalseNegatives(thresholds=config.THRESHOLD, name='fn'),
            AUC(name='auc'),
            AUC(name='prc', curve='PR')
        ]
    )
    return model

In [ ]:
model = build_model()

In [ ]:
model.summary()

In [ ]:
labels = np.asarray(y_train).reshape(-1)
final_epochs = config.EPOCHS

if config.VALIDATION:
    histories = []
    best_epochs = []
    kfold = StratifiedKFold(n_splits=config.N_SPLITS, shuffle=True, random_state=config.SEED)
    for fold, (train, test) in enumerate(kfold.split(X_train_raw, labels), start=1):
        fold_scaler = MinMaxScaler(feature_range=(0, config.FEATURE_RANGE_MAX))
        X_fold_train = fold_scaler.fit_transform(take_rows(X_train_raw, train))
        X_fold_val = fold_scaler.transform(take_rows(X_train_raw, test))
        y_fold_train = np.asarray(take_rows(y_train, train)).reshape(-1)
        y_fold_val = np.asarray(take_rows(y_train, test)).reshape(-1)

        fold_model = build_model()
        history = fold_model.fit(
            x=X_fold_train,
            y=y_fold_train,
            batch_size=config.BATCH_SIZE,
            epochs=config.EPOCHS,
            verbose=config.VERBOSE,
            validation_data=(X_fold_val, y_fold_val),
            callbacks=[
                EarlyStopping(monitor='val_loss', mode='min', patience=5, restore_best_weights=True),
                TensorBoard(log_dir=os.path.join(config.LOG_DIR, f'fold_{fold}'))
            ]
        )
        histories.append(history)

        val_loss = np.asarray(history.history.get('val_loss', []), dtype=np.float64)
        if val_loss.size == 0:
            raise ValueError(f'Fold {fold}: no validation loss was recorded')
        if not np.all(np.isfinite(val_loss)):
            raise ValueError(f'Fold {fold}: validation loss contains non-finite values')

        best_epoch = int(np.argmin(val_loss)) + 1
        best_epochs.append(best_epoch)
        print(f'Fold {fold}/{config.N_SPLITS}: best epoch {best_epoch} of {len(val_loss)} trained (val_loss {val_loss[best_epoch - 1]:.4f})')
        if len(val_loss) >= config.EPOCHS:
            print(f'Warning: fold {fold}/{config.N_SPLITS} exhausted the epoch cap ({config.EPOCHS}) without early stopping; epoch search is capped')

    mean_epochs = float(np.mean(best_epochs))
    final_epochs = max(1, int(np.ceil(mean_epochs)))
    print(f'Best epochs per fold: {best_epochs} - Mean: {mean_epochs:.2f} - Chosen final epochs: {final_epochs}')
else:
    print(f'No epoch search; using configured {final_epochs} epochs for final training')

scaler = MinMaxScaler(feature_range=(0, config.FEATURE_RANGE_MAX))
X_train = pd.DataFrame(scaler.fit_transform(X_train_raw)).values

model = build_model()
final_history = model.fit(
    x=X_train,
    y=labels,
    batch_size=config.BATCH_SIZE,
    epochs=final_epochs,
    verbose=config.VERBOSE,
    callbacks=[
        TensorBoard(log_dir=os.path.join(config.LOG_DIR, 'final'))
    ]
)

X_test = pd.DataFrame(scaler.transform(X_test_raw)).values

scale = scaler.scale_.astype(np.float64)
min_  = scaler.min_.astype(np.float64)

assert scale.shape == min_.shape
assert scale.ndim == 1

with open("scaler.bin", "wb") as f:
    np.array([len(scale)], dtype=np.int32).tofile(f)
    scale.tofile(f)
    min_.tofile(f)

In [ ]:
essentials = ['loss', 'prc', 'precision', 'recall']
confusions = ['tp', 'tn', 'fp', 'fn']
if config.VALIDATION:
    plot_cross_val_essential_metrics(essentials, histories)
    plot_cross_val_confusion_metrics(confusions, histories)
else:
    plot_essential_metrics(essentials, final_history)
    plot_confusion_metrics(confusions, final_history)

In [ ]:
scores = model.evaluate(X_test, y_test, verbose=config.VERBOSE)
print(f'Loss: {scores[0] * 100:.2f}% - PRC: {scores[9] * 100:.2f}% - Precision: {scores[3] * 100:.2f}% - Recall: {scores[2] * 100:.2f}%')

In [ ]:
spec = [tf.TensorSpec([None, config.NUM_FEATURES], tf.float32, name="input")]

@tf.function(input_signature=spec)
def model_fn(x):
    return model(x, training=False)

onnx_model, _ = tf2onnx.convert.from_function(model_fn, input_signature=spec, opset=18)
onnx.checker.check_model(onnx_model)
onnx.save(onnx_model, "model.onnx")

In [ ]:
y_pred = (model.predict(X_test) > config.THRESHOLD).astype('int32')
cm = confusion_matrix(y_true=y_test, y_pred=y_pred)
fig = plt.figure(figsize=[8, 6])
ax = fig.add_subplot(111)
ConfusionMatrixDisplay(confusion_matrix=cm).plot(ax=ax)
ax.set_title('Confusion Matrix')
plt.savefig(os.path.join('figures', 'learning', config.DATASET, 'confusion_matrix.png'), bbox_inches='tight')

In [ ]:
y_train_pred = model.predict(X_train, verbose=config.VERBOSE)
y_test_pred = model.predict(X_test, verbose=config.VERBOSE)
fig = plt.figure(figsize=[8, 6])
ax = fig.add_subplot(111)
ax.set_aspect('equal', adjustable='datalim')
plot_prc('Train Baseline', y_train, y_train_pred, color='C0')
plot_prc('Test Baseline', y_test, y_test_pred, color='C1', linestyle='--')
plt.title('Precision-Recall Curve')
plt.legend(loc='lower left');
fig.savefig(os.path.join('figures', 'learning', config.DATASET, 'metrics-prc.png'), bbox_inches='tight')

In [ ]:
dd = {}
dd = load_data(dd, os.path.join('data', 'fuzzing', 'original.zip'))

In [ ]:
df = pd.DataFrame.from_dict(dd, orient='index').add_prefix('f')
df.shape

In [ ]:
if len(df.index) > 0:
    X_pred = df.drop('f0', axis=1).values
    X_pred = pd.DataFrame(scaler.transform(X_pred))

In [ ]:
raw_scores = []
if len(df.index) > 0:
    raw_scores = [score for row in model.predict(X_pred, verbose=config.VERBOSE) for score in row]

if not np.all(np.isfinite(raw_scores)):
    raise ValueError('Prediction scores contain non-finite values')
ranked = sorted(range(len(raw_scores)), key=lambda i: (-raw_scores[i], df.index[i]))
above = [i for i in ranked if raw_scores[i] > config.THRESHOLD]
indices = (above or ranked)[:20]

In [ ]:
lst = [df.index[i] for i in indices]

print(f'Selected {len(lst)} files out of {len(df.index)} total:')
for filename in lst:
    print(filename)